# Imports and function defs

In [7]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cna, glob, os, gc
import vima
%run ../common.py

In [8]:
def test_clusters(df, cols, pheno, donor, Nnull=1000):
    X = df[cols].values.copy()
    pheno = df[pheno].copy()
    donorids = df[donor].copy()
    
    filter = np.isfinite(pheno)
    X = X[filter,:]
    pheno = pheno[filter]
    donorids = donorids[filter]
    
    X = (X - X.mean(axis=0))/X.std(axis=0)
    y_ = cna.tl._stats.grouplevel_permutation(donorids, pheno, Nnull).astype('float')
    pheno = (pheno - pheno.mean())/pheno.std()
    y_ -= y_.mean(axis=0)
    y_ /= y_.std(axis=0)
    
    ncorrs = np.nan_to_num(X.T.dot(pheno) / len(X))
    nullncorrs = np.nan_to_num(X.T.dot(y_) / len(X))
    pvals = ((np.abs(nullncorrs) >= np.abs(ncorrs)[:,None]).sum(axis=1) + 1)/(Nnull + 1)
    globalp = (((ncorrs**2).sum() <= (nullncorrs**2).sum(axis=0)).sum() + 1)/(Nnull + 1)
    
    maxcorr = max(np.abs(ncorrs).max(), 0.001)
    fdr_thresholds = np.arange(maxcorr/4, maxcorr, maxcorr/400)
    fdr_vals = cna.tl._stats.empirical_fdrs(ncorrs, nullncorrs, fdr_thresholds)

    fdrs = pd.DataFrame({
        'threshold':fdr_thresholds,
        'fdr':fdr_vals,
        'num_detected': [(np.abs(ncorrs)>t).sum() for t in fdr_thresholds]})
    if len(fdrs[fdrs.fdr <= 0.1]) > 0:
        fdr10pt = fdrs[fdrs.fdr <= 0.1].threshold.min()
    else:
        fdr10pt = np.infty
    return fdrs, fdr10pt, ncorrs, pvals, globalp

In [9]:
from scipy.stats import entropy
def integration(d):
    A = d.obsp['connectivities']
    A /= A.sum(axis=1)
    S = pd.get_dummies(d.obs.sid).astype(np.float32)
    baseline = np.power(2, entropy(d.obs.sid.value_counts() / len(d), base=2))
    perplexities = np.power(2, entropy(np.array(A.dot(S)), axis=1, base=2)) / baseline
    d.obs['perplexity'] = perplexities

def test_cluster_cc(d, samplemeta, secondary_pheno=None):
    # Determine cluster key
    if 'cluster_method' in d.obs.columns:
        cluster_key = 'cluster_method'
    elif 'leiden_1' in d.obs.columns:
        cluster_key = 'leiden_1'
    else:
        cluster_key = [c for c in d.obs.columns if c.startswith('leiden')][-1]
    print(f'Using {cluster_key} for clustering. There are {d.obs[cluster_key].nunique()} clusters')

    # Build crosstab and normalize
    ct = pd.crosstab(d.obs['sid'], d.obs[cluster_key]).div(
        pd.crosstab(d.obs['sid'], d.obs[cluster_key]).sum(axis=1), axis=0)
    ct.index.name = 'sid'
    clusts = ct.columns.values
    
    if secondary_pheno:
        ct['case2'] = samplemeta[secondary_pheno]

    # Helper to run test_clusters and store results
    def store_results(suffix, pheno, mask=None):
        cols = clusts if mask is None else clusts[mask]
        if len(cols) > 0:
            myct = ct[cols].div(ct[cols].sum(axis=1), axis=0)
            myct['donor'] = samplemeta.donor
            myct[pheno] = samplemeta[pheno]
            fdrs, fdr10pt, stats, ps, globalp = test_clusters(myct, cols, pheno, 'donor', Nnull=10000)
            d.uns[f'clustercc{suffix}'] = pd.DataFrame({'corr':stats, 'p':ps}, index=pd.Series(cols, name='cluster'))
            d.uns[f'clustercc{suffix}_key'] = cluster_key
            d.uns[f'clustercc{suffix}_minp'] = np.min(ps*len(ps))
            d.uns[f'clustercc{suffix}_globalp'] = globalp
            d.uns[f'clustercc{suffix}_npos'] = d.obs[cluster_key].isin(cols[stats > fdr10pt]).sum()
            d.uns[f'clustercc{suffix}_nneg'] = d.obs[cluster_key].isin(cols[stats < -fdr10pt]).sum()
            return stats, fdr10pt

    # Main phenotype
    stats, fdr10pt = store_results('', 'case')

    # Secondary phenotype (if provided)
    mask = stats > fdr10pt
    if secondary_pheno and mask.sum() > 0:
        store_results('2', secondary_pheno, mask)
    else:
        store_results('2', secondary_pheno, mask=np.zeros(len(clusts), dtype=bool))

def test_mn_cc(d, samplemeta, secondary_pheno=None):
    def store_results(suffix, pheno, mask):
        myd = d[mask].copy() if mask is not None else d
        if len(myd) > 100:
            if len(myd) < len(d):
                sc.pp.neighbors(myd)
            d.uns[f'mncc{suffix}_p'], D = vima.association([myd], samplemeta[pheno], 'sid', donorids=samplemeta.donor,
                                                key_added=f'mncoef{suffix}', make_umap=False, allow_low_sample_size=True)
            d.obs[f'mncoef{suffix}_fdr'] = D.obs.mncoef_fdr
            d.obs[f'mncoef{suffix}'] = D.obs.mncoef
            d.uns[f'mncc{suffix}_npos'] = ((d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef > 0)).sum()
            d.uns[f'mncc{suffix}_nneg'] = ((d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef < 0)).sum()
    
    store_results('', 'case', None)

    mask = (d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef > 0)
    if secondary_pheno and mask.sum() > 0:
        print(mask.sum(), 'positive correlations for primary phenotype')
        store_results('2', secondary_pheno, mask)

In [10]:
def assess(dsetname, samplemeta, secondary_pheno=None, suffix='*'):
    embeddings = glob.glob(f'_embeddings/{dsetname}_*_{suffix}.h5ad')
    for embedding in embeddings:
        fname = os.path.basename(embedding)
        method = fname.split('_')[1]
        harm = fname.split('_')[2]
        print(method, harm)
        d = sc.read_h5ad(embedding)
        integration(d)
        test_cluster_cc(d, samplemeta, secondary_pheno=secondary_pheno)
        test_mn_cc(d, samplemeta, secondary_pheno=secondary_pheno)
        print(method, harm)
        print(f'\tmed perp: {d.obs.perplexity.median()}')
        print(f'\tcluster minp: {d.uns['clustercc_minp']}, cluster globalp: {d.uns['clustercc_globalp']}, npos: {d.uns['clustercc_npos']} nneg: {d.uns['clustercc_nneg']}')
        print(f'\tMN p: {d.uns['mncc_p']}, npos: {d.uns['mncc_npos']} nneg: {d.uns['mncc_nneg']}')
        if 'clustercc2_minp' in d.uns:
            print(f'\tcluster2 minp: {d.uns['clustercc2_minp']}, cluster2 globalp: {d.uns['clustercc2_globalp']}')
        if 'mncc2_p' in d.uns:
            print(f'\tMN2 p: {d.uns['mncc2_p']}, npos: {d.uns['mncc2_npos']} nneg: {d.uns['mncc2_nneg']}')
        print('======')
        d.write(f'_results/{fname}')

# Run

## ALZ

In [5]:
# generate samplemeta with one row per sample (rather than per donor)
cells = pd.read_csv('../../ALZ/alz-data/SEAAD_MTG_MERFISH_metadata.2024-05-03.noblanks.harmonized.txt',
                         sep='\t')
cells['donor'] = cells.index.str.split('_').str[0]
cells['sid'] = cells.index.str.split('_').str[1]
sid_to_donor = cells[['sid', 'donor']].drop_duplicates()

samplemeta = pd.read_csv('../../ALZ/alz-data/sea-ad_cohort_donor_metadata_encoded_20240924.tsv',
                         sep='\t').drop(columns=['Donor ID']).set_index('donor', drop=True)
samplemeta = pd.merge(sid_to_donor, samplemeta, left_on='donor', right_index=True, how='left').set_index('sid', drop=True)
samplemeta['case'] = samplemeta['Consensus Clinical Dx (choice=Control)'] != 'Checked'

In [6]:
assess('ALZ', samplemeta)

utag harm.h5ad
Using leiden_0.3 for clustering. There are 16 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
There are 1652992 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.36676332366763326
utag harm.h5ad
	med perp: 0.1360127329826355
	cluster minp: 0.8591140885911409, cluster globalp: 0.37096290370962903, npos: 0 nneg: 0
	MN p: 0.36676332366763326, npos: 0 nneg: 0
stagate harm.h5ad
Using leiden_1 for clustering. There are 24 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
There are 200115 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.8063193680631937
stagate harm.h5ad
	med perp: 0.09967397153377533
	cluster minp: 0.395960403959604, cluster globalp: 0.9026097390260974, npos: 0 nneg: 0
	MN p: 0.8063193680631937, npos: 0 nneg: 0
cellcharter harm.h5ad
Using cluster_method for clustering. There are 4 clusters


...

computing MAT
There are 1294316 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.5596440355964404
tissuemosaic harm.h5ad
	med perp: 0.18052935600280762
	cluster minp: 4.199380061993801, cluster globalp: 0.7018298170182982, npos: 0 nneg: 0
	MN p: 0.5596440355964404, npos: 0 nneg: 0
patchcelltypeabundance harm.h5ad
Using leiden1 for clustering. There are 128 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
performing association test
P = 0.042695730426957304
patchcelltypeabundance harm.h5ad
	med perp: 0.14453168213367462
	cluster minp: 1.0366963303669634, cluster globalp: 0.08919108089191082, npos: 0 nneg: 0
	MN p: 0.042695730426957304, npos: 0 nneg: 0
patchavgmm harm.h5ad
Using leiden1 for clustering. There are 20 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
performing association test
P = 0.035396460353964605
patchavgmm harm.h5ad
	med perp: 0.14184466004371643
	cluster minp: 0.031996800319968, cluster globalp: 0.0392960703929607, npos: 0 nneg: 0
	MN p: 0.035396460353964605, npos: 0 nneg: 0


## RA

In [11]:
# read in and reformat sample metadata
fullmeta = pd.read_csv('../../RA/BHAM-data/ihc-metadata.csv').set_index('subject_id')[['CTAP']]
fullmeta.index = fullmeta.index.str.replace('V0', '') # reformat sample names
fullmeta['fstar'] = (fullmeta.CTAP == 'F') | (fullmeta.CTAP == 'T + F') | (fullmeta.CTAP == 'E + F + M') # define our phenotype

# change samplemeta so that each row is a sample rather than a donor
inourdata = sc.read_h5ad('_embeddings/RA_stagate_noharm.h5ad').obs[['sid','donor']].drop_duplicates()
samplemeta = pd.merge(inourdata, fullmeta, left_on='donor', right_index=True, how='left').set_index('sid', drop=True)
samplemeta['case'] = samplemeta.fstar

In [12]:
assess('RA', samplemeta)

canvas noharm.h5ad
Using leiden_1 for clustering. There are 36 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
performing association test
P = 0.1464853514648535
canvas noharm.h5ad
	med perp: 0.16841016709804535
	cluster minp: 0.11518848115188482, cluster globalp: 0.16108389161083891, npos: 0 nneg: 0
	MN p: 0.1464853514648535, npos: 0 nneg: 0
patchavgmm noharm.h5ad
Using leiden1 for clustering. There are 37 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
performing association test
P = 0.0023997600239976003
patchavgmm noharm.h5ad
	med perp: 0.1218021810054779
	cluster minp: 0.603039696030397, cluster globalp: 0.035896410358964105, npos: 0 nneg: 0
	MN p: 0.0023997600239976003, npos: 0 nneg: 0
stagate noharm.h5ad
Using leiden_1 for clustering. There are 74 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
There are 366976 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.0192980701929807
stagate noharm.h5ad
	med perp: 0.06294918060302734
	cluster minp: 1.3096690330966905, cluster globalp: 0.017298270172982702, npos: 16047 nneg: 18250
	MN p: 0.0192980701929807, npos: 0 nneg: 0


## UC

In [9]:
# read in sample metadata
samplemeta = pd.read_csv('../../UC/UC-data/2024_10_16_UC_Patient_Metadata.csv').rename(columns={'NEW Label':'sid', 'Patient.ID':'donor'}).set_index('sid', drop=True)
samplemeta.donor = samplemeta.donor.astype('str')
samplemeta['case'] = (samplemeta.Status == 'UC').astype('float')
samplemeta.loc[(samplemeta.TNFnow == 'y') | (samplemeta.TNFprior == 'y'), 'case'] = np.nan
samplemeta['TNF'] = (samplemeta.TNFprior == 'y').astype('float')
samplemeta.loc[samplemeta.case == 0, 'TNF'] = np.nan

In [10]:
assess('UC', samplemeta, secondary_pheno='TNF')

canvas harm.h5ad
Using leiden_1 for clustering. There are 16 clusters


/var/folders/w_/x2_v44_93nq_b3svp199t1rw0000gn/T/ipykernel_87502/797230936.py:11: RuntimeWarning: invalid value encountered in divide
  X = (X - X.mean(axis=0))/X.std(axis=0)
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
performing association test
P = 0.025697430256974303
canvas harm.h5ad
	med perp: 0.12142954021692276
	cluster minp: 0.009599040095990401, cluster globalp: 0.008899110088991101, npos: 0 nneg: 306
	MN p: 0.025697430256974303, npos: 0 nneg: 232
patchcelltypeabundance harm.h5ad
Using leiden1 for clustering. There are 80 clusters


/var/folders/w_/x2_v44_93nq_b3svp199t1rw0000gn/T/ipykernel_87502/797230936.py:11: RuntimeWarning: invalid value encountered in divide
  X = (X - X.mean(axis=0))/X.std(axis=0)
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:79: RuntimeWarning: invalid value encountered in divide
  fdp = tails / ranks


computing MAT
performing association test
P = 0.006199380061993801
1 positive correlations for primary phenotype
patchcelltypeabundance harm.h5ad
	med perp: 0.13611429929733276
	cluster minp: 0.03999600039996, cluster globalp: 0.0282971702829717, npos: 1232 nneg: 4414
	MN p: 0.006199380061993801, npos: 1 nneg: 1017
	cluster2 minp: 1.0, cluster2 globalp: 1.0
utag harm.h5ad
Using leiden_0.3 for clustering. There are 69 clusters


...

computing MAT
performing association test
P = 0.00019998000199980003
11 positive correlations for primary phenotype
patchavgmm harm.h5ad
	med perp: 0.1651913821697235
	cluster minp: 0.0206979302069793, cluster globalp: 0.0016998300169983002, npos: 781 nneg: 1958
	MN p: 0.00019998000199980003, npos: 11 nneg: 604
	cluster2 minp: 1.0, cluster2 globalp: 1.0
stagate harm.h5ad
Using leiden_1 for clustering. There are 45 clusters


/var/folders/w_/x2_v44_93nq_b3svp199t1rw0000gn/T/ipykernel_87502/797230936.py:11: RuntimeWarning: invalid value encountered in divide
  X = (X - X.mean(axis=0))/X.std(axis=0)
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:79: RuntimeWarning: invalid value encountered in divide
  fdp = tails / ranks


computing MAT
performing association test
P = 0.011898810118988102
stagate harm.h5ad
	med perp: 0.0880926176905632
	cluster minp: 0.09449055094490551, cluster globalp: 0.016198380161983803, npos: 3936 nneg: 2301
	MN p: 0.011898810118988102, npos: 0 nneg: 401
	cluster2 minp: 1.0, cluster2 globalp: 1.0
tissuemosaic harm.h5ad
Using leiden1 for clustering. There are 221 clusters


/var/folders/w_/x2_v44_93nq_b3svp199t1rw0000gn/T/ipykernel_87502/797230936.py:11: RuntimeWarning: invalid value encountered in divide
  X = (X - X.mean(axis=0))/X.std(axis=0)
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAT
There are 1536983 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.28247175282471754
tissuemosaic harm.h5ad
	med perp: 0.026155002415180206
	cluster minp: 0.06629337066293371, cluster globalp: 0.09219078092190781, npos: 0 nneg: 204652
	MN p: 0.28247175282471754, npos: 0 nneg: 856


# See results

In [12]:
collate_cc('ALZ', D=sc.read_h5ad('../ALZ/_results/cc_dementia.h5ad'))

,microniche,harmonized,bonferroni,p,npos,nneg,ntotal,frac_pos,frac_neg
method,,,,,,,,,
vima,NaN,NaN,NaN,0.005399,22953.0,17204.0,70898,0.323747,0.242658
patchavgmm,False,True,False,0.039296,0.0,0.0,70898,0.000000,0.000000
patchavgmm,False,True,False,0.039296,0.0,0.0,70898,0.000000,0.000000
patchavgmm,False,True,False,0.039296,0.0,0.0,70898,0.000000,0.000000
patchavgmm,False,True,False,0.039296,0.0,0.0,70898,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
stagate,False,True,False,0.902610,0.0,0.0,200115,0.000000,0.000000
stagate,False,True,False,0.902610,0.0,0.0,200115,0.000000,0.000000
stagate,False,True,False,0.902610,0.0,0.0,200115,0.000000,0.000000


In [18]:
collate_cc('RA', D=sc.read_h5ad('../RA/_results/cc_fstar.h5ad'))

canvas noharm
patchavgmm noharm
stagate noharm
stagate harm
patchavgmm harm
canvas harm


,microniche,harmonized,bonferroni,p,npos,nneg,ntotal,frac_pos,frac_neg
method,,,,,,,,,
vima,NaN,NaN,NaN,0.000200,2089.0,3133.0,10345,0.201933,0.302852
patchavgmm,True,False,False,0.002400,0.0,0.0,22497,0.000000,0.000000
patchavgmm,False,True,False,0.006799,2872.0,2563.0,22497,0.127661,0.113926
stagate,False,False,False,0.017298,16047.0,18250.0,366976,0.043728,0.049731
stagate,True,False,False,0.019298,0.0,0.0,366976,0.000000,0.000000
stagate,False,True,False,0.025997,0.0,0.0,366976,0.000000,0.000000
patchavgmm,False,False,False,0.035896,0.0,0.0,22497,0.000000,0.000000
canvas,False,False,True,0.100790,NaN,NaN,20938,NaN,NaN
canvas,True,False,False,0.146485,0.0,0.0,20938,0.000000,0.000000


In [19]:
collate_cc('UC', D=sc.read_h5ad('../UC/_results/cc_uc.h5ad'))

stagate noharm
patchavgmm noharm
patchcelltypeabundance noharm
tissuemosaic noharm
canvas noharm
utag noharm
canvas harm
patchcelltypeabundance harm
utag harm
patchavgmm harm
stagate harm
tissuemosaic harm


,microniche,harmonized,bonferroni,p,npos,nneg,ntotal,frac_pos,frac_neg
method,,,,,,,,,
vima,NaN,NaN,NaN,0.001700,4146.0,7693.0,21359,0.194110,0.360176
patchavgmm,False,True,False,0.001700,781.0,1958.0,21359,0.036565,0.091671
patchavgmm,False,False,False,0.002700,0.0,1551.0,21359,0.000000,0.072616
patchavgmm,True,False,False,0.003100,1.0,268.0,21359,0.000047,0.012547
canvas,False,True,False,0.008899,0.0,306.0,2101,0.000000,0.145645
utag,False,False,False,0.011199,0.0,413071.0,1647673,0.000000,0.250700
patchavgmm,False,False,True,0.015998,NaN,NaN,21359,NaN,NaN
stagate,False,True,False,0.016198,3936.0,2301.0,51408,0.076564,0.044760
utag,False,True,False,0.018898,0.0,133269.0,1647673,0.000000,0.080883


In [23]:
collate_cc('UC', suffix='2', D=sc.read_h5ad('../UC/_results/cc_tnf.h5ad'))

stagate noharm
patchavgmm noharm
patchcelltypeabundance noharm
tissuemosaic noharm
canvas noharm
utag noharm
canvas harm
patchcelltypeabundance harm
utag harm
patchavgmm harm
stagate harm
tissuemosaic harm


,microniche,harmonized,bonferroni,p,npos,nneg,ntotal,frac_pos,frac_neg
method,,,,,,,,,
vima,NaN,NaN,NaN,0.0015,1317,990,4146,0.317656,0.238784
patchcelltypeabundance,False,True,False,1.0000,0,0,21359,0.000000,0.000000
patchavgmm,False,True,False,1.0000,0,0,21359,0.000000,0.000000
stagate,False,True,False,1.0000,0,0,51408,0.000000,0.000000


# Collate all results and write to file

In [5]:
allresults = []
for dset, vimaD, pheno, suff in [
    ('RA', '../RA/_results/cc_fstar.h5ad', 'RA', ''),
    ('UC', '../UC/_results/cc_uc.h5ad', 'UC', ''),
    ('UC', '../UC/_results/cc_tnf.h5ad', 'TNFi', '2'),
    ('ALZ', '../ALZ/_results/cc_dementia.h5ad', 'Dementia', '')
    ]:
    myresults = collate_cc(dset, D=sc.read_h5ad(vimaD), suffix=suff)
    myresults['pheno'] = pheno
    allresults.append(myresults)
    gc.collect()
results = pd.concat(allresults)

canvas noharm
patchavgmm noharm
stagate noharm
stagate harm
patchavgmm harm
canvas harm
stagate noharm
patchavgmm noharm
patchcelltypeabundance noharm
tissuemosaic noharm
canvas noharm
utag noharm
canvas harm
patchcelltypeabundance harm
utag harm
patchavgmm harm
stagate harm
tissuemosaic harm
stagate noharm
patchavgmm noharm
patchcelltypeabundance noharm
tissuemosaic noharm
canvas noharm
utag noharm
canvas harm
patchcelltypeabundance harm
utag harm
patchavgmm harm
stagate harm
tissuemosaic harm
patchavgmm noharm
stagate noharm
tissuemosaic noharm
patchcelltypeabundance noharm
utag noharm
cellcharter noharm
canvas noharm
utag harm
stagate harm
cellcharter harm
canvas harm
tissuemosaic harm
patchcelltypeabundance harm
patchavgmm harm


In [6]:
results['dataset'] = results.pheno.map({'RA': 'RA subtypes',
                              'UC':'UC vs healthy',
                              'TNFi':'TNFi vs no',
                              'Dementia':'Dementia vs control'})
results.to_csv('_results/allbenchmarking.csv')